# Bias Analysis

Survivorship, analyst optimism, recency bias, and shrinkage.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_bridge import load_all_signal_inputs
from src.signal_diagnostics import compute_ic_timeseries, ic_summary, compute_turnover
from src.bias_correction import (
    check_survivorship, correct_analyst_optimism,
    detect_recency_bias, shrink_weights,
)
from src.combination import combine_ols
from src.walkforward import walkforward_evaluate
from src.signals import signal_earnings_revision, rank_normalize

## Shared setup: load panel and outcome

In [ ]:
panel = pd.read_parquet("../data/processed/signal_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])
sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]

data = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")
prices      = data["prices"]
earnings_df = data["earnings_estimates"]

# Quarter-end forward returns (outcome)
wide    = prices.pivot(index="date", columns="ticker", values="adj_close")
qprices = wide.resample("QE").last()
fwd     = qprices.shift(-1) / qprices - 1
outcome_df = fwd.stack().reset_index()
outcome_df.columns = ["date", "ticker", "outcome"]
outcome_df = outcome_df.dropna()

# Panel filtered to quarter-end dates, with outcome merged in
quarter_ends = set(outcome_df["date"].unique())
panel_q = panel[panel["date"].isin(quarter_ends)].copy()
panel_q = panel_q.merge(outcome_df, on=["ticker", "date"], how="inner")

print(f"Panel with outcome: {panel_q.shape}")

## 1. Survivorship bias check

In [ ]:
# Synthetic universe: 100 tickers, 20 were 'removed' before panel end.
# Our data_bridge includes all 100 (removed and surviving), so coverage = 100%.
rng       = np.random.default_rng(0)
all_ticks = panel["ticker"].unique()
panel_start = panel["date"].min()
panel_end   = panel["date"].max()

removed_tickers = rng.choice(all_ticks, size=20, replace=False)
# Removed firms delisted at a random date before panel end
removed_dates = pd.to_datetime([
    str(rng.integers(2015, 2023)) + "-06-30" for _ in removed_tickers
])
universe_dates = pd.DataFrame({
    "ticker":       all_ticks,
    "added_date":   panel_start,
    "removed_date": pd.NaT,
})
for t, d in zip(removed_tickers, removed_dates):
    universe_dates.loc[universe_dates["ticker"] == t, "removed_date"] = d
universe_dates["removed_date"] = universe_dates["removed_date"].fillna(panel_end)

stats = check_survivorship(panel, universe_dates)
print("Survivorship audit:")
for k, v in stats.items():
    print(f"  {k}: {v:.2%}" if isinstance(v, float) else f"  {k}: {v}")

print("\nTakeaway: our synthetic panel includes all delisted firms "
      "(coverage = 100%); with real data, missing delistings inflate returns "
      "because losers that vanish are excluded.")

## 2. Analyst optimism correction

In [ ]:
estimates = earnings_df[["ticker", "date", "estimate"]].copy()
actuals   = earnings_df[["ticker", "date", "actual"]].copy()

corrected = correct_analyst_optimism(estimates, actuals, window=20)

# Raw revision signal: (actual - estimate) / |estimate|
def make_revision(est_col, df=earnings_df):
    merged = df[["ticker", "date", "actual"]].merge(
        est_col, on=["ticker", "date"]
    )
    denom = merged.iloc[:, -1].abs().clip(lower=0.01)
    out = merged[["ticker", "date"]].copy()
    out["signal"] = (merged["actual"] - merged.iloc[:, -1]) / denom
    out["signal"] = out.groupby("date")["signal"].rank(pct=True)
    return out

raw_rev = make_revision(earnings_df[["ticker", "date", "estimate"]])
cor_rev = make_revision(corrected.rename(columns={"estimate_corrected": "estimate"}))

ic_raw = compute_ic_timeseries(raw_rev, outcome_df)
ic_cor = compute_ic_timeseries(cor_rev, outcome_df)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ic_raw.index, ic_raw.rolling(4).mean(), label="Raw revision (4q MA)",    color="tomato")
ax.plot(ic_cor.index, ic_cor.rolling(4).mean(), label="Corrected revision (4q MA)", color="steelblue")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Revision signal IC: raw vs optimism-corrected")
ax.legend()
plt.tight_layout()
plt.show()

sr, sc = ic_summary(ic_raw), ic_summary(ic_cor)
print(f"Raw IC:       mean={sr['mean_ic']:+.4f}  IR={sr['ir']:+.3f}")
print(f"Corrected IC: mean={sc['mean_ic']:+.4f}  IR={sc['ir']:+.3f}")
print("\nTakeaway: correcting for the systematic 5% upward bias shifts estimates "
      "downward, changing what counts as a 'positive surprise' and altering "
      "the signal's IC — with real data this correction meaningfully improves IR.")

## 3. Recency bias detection — top 3 signals by IC

In [ ]:
# Recompute IC for all signals to pick top 3
ic_all = {}
for col in sig_cols:
    sig_df = panel_q[["ticker", "date", col]].dropna().rename(columns={col: "signal"})
    if len(sig_df) >= 100:
        ic_all[col] = compute_ic_timeseries(sig_df, outcome_df)

top3 = sorted(ic_all, key=lambda c: ic_all[c].mean(), reverse=True)[:3]
print(f"Top 3 signals by mean IC: {top3}")

fig, axes = plt.subplots(3, 1, figsize=(12, 10))
for ax, col in zip(axes, top3):
    rec = detect_recency_bias(ic_all[col])
    ax.plot(rec.index, rec["ic_short"], label="Short-window IC (8q)",  color="steelblue")
    ax.plot(rec.index, rec["ic_long"],  label="Long-window IC (40q)",  color="navy", linestyle="--")
    flag_dates = rec.index[rec["flag"]]
    ax.scatter(flag_dates, rec.loc[flag_dates, "ic_short"],
               color="red", zorder=5, s=30, label="Divergence flag")
    ax.axhline(0, color="black", linewidth=0.7, linestyle=":")
    ax.set_title(f"{col}  —  short vs long IC window")
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle("Recency bias: short vs long IC window (red = divergence)", fontsize=11)
plt.tight_layout()
plt.show()

print("Takeaway: red dots mark periods where a recency-biased weighting scheme "
      "would over- or under-weight a signal based on a streak that subsequently "
      "mean-reverts — the long-window IC is the more reliable anchor.")

## 4. Shrinkage: OLS weights shrunk toward equal weight

In [ ]:
# Use signals with full quarter-end coverage for a clean walkforward
coverage = panel_q[sig_cols].notna().mean()
use_cols = coverage[coverage > 0.5].index.tolist()
print(f"Signals used (coverage > 50%): {use_cols}")

shrinkage_levels = [0.0, 0.25, 0.5, 0.75, 1.0]
shrink_results = []

for s in shrinkage_levels:
    def make_fn(shrinkage=s):
        def fn(panel_in, signal_cols, outcome_col="outcome"):
            raw = combine_ols(panel_in, signal_cols, outcome_col=outcome_col)
            return shrink_weights(raw, shrinkage=shrinkage)
        return fn

    res = walkforward_evaluate(
        panel_q, use_cols, "outcome",
        combination_fn=make_fn(s),
        min_train_periods=12,
    )
    mean_ic = res["composite_ic"].mean() if len(res) else float("nan")
    shrink_results.append({"shrinkage": s, "mean_oos_ic": mean_ic,
                            "n_periods": len(res)})
    print(f"  shrinkage={s:.2f}  mean_oos_ic={mean_ic:+.4f}  periods={len(res)}")

shrink_df = pd.DataFrame(shrink_results)
print("\n" + shrink_df.to_string(index=False,
    formatters={"shrinkage": "{:.2f}".format, "mean_oos_ic": "{:+.4f}".format}))
print("\nTakeaway: shrinkage toward equal weight acts as regularization — "
      "on synthetic random data OLS weights are pure noise, so higher shrinkage "
      "(more equal-weight) should produce the flattest and most stable OOS IC; "
      "with real predictive signals the optimal shrinkage is typically 0.5–0.75.")